# 🏗️ Notebook 1: Typeahead / Autocomplete — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

As the user types `"goo"` → we suggest `"google", "google docs", "good morning"`.
Under **~20 ms per keystroke**, for millions of users.

Think Google search box, YouTube search, Amazon product search, VS Code command palette.

### Functional requirements
- Given a **prefix**, return **top-K suggestions** ordered by popularity.
- Suggestions stay fresh as new queries trend.
- (Nice-to-have, later) personalization, typo tolerance, multi-language.

### Non-functional
- **Very low latency**: p99 under ~20 ms (because *every keystroke* hits the service).
- **High QPS**: a power-user can type 5–10 chars/sec. One search = many requests.
- **Graceful**: handle weird input (spaces, accents, emoji, typos) without crashing.


## Back-of-the-envelope: why is this hard?

Let's do quick math. It's the single most useful skill in system design interviews.


In [ ]:
# 🔢 Capacity estimate — run me!
DAU = 300_000_000                 # daily active users, Google-ish
searches_per_user_per_day = 5
chars_typed_per_search = 15       # avg query length a user types out
requests_per_day = DAU * searches_per_user_per_day * chars_typed_per_search

seconds_per_day = 24 * 3600
avg_qps = requests_per_day / seconds_per_day
peak_qps = avg_qps * 3            # peak is ~3x average for consumer apps

print(f"Requests/day: {requests_per_day:_}")
print(f"Average QPS : {avg_qps:_.0f}")
print(f"Peak QPS    : {peak_qps:_.0f}")
print()

# --- The saving grace: prefix traffic is wildly skewed --------------------
# A 15-char query issues 15 requests, but 1- and 2-char prefixes are shared by
# *everybody*. Those few thousand hot prefixes are trivially cacheable at the edge.
# Model it crudely: assume the first 3 keystrokes of every search are edge hits.
cacheable_fraction = 3 / chars_typed_per_search
origin_qps = peak_qps * (1 - cacheable_fraction)
print(f"Edge-cacheable share : {cacheable_fraction:.0%}  (prefixes of length 1-3)")
print(f"Peak QPS at origin   : {origin_qps:_.0f}")
print(f"Replicas at 1k QPS/box: ~{origin_qps/1000:_.0f}  (vs ~{peak_qps/1000:_.0f} with no edge cache)")
print()

# --- How big is the serving index? ---------------------------------------
# This is the number that decides "one box or many". Do it before you commit
# to "just keep the trie in memory".
distinct_terms  = 100_000_000     # distinct queries worth suggesting
avg_term_chars  = 12
alphabet        = 26
# Nodes ~= one per distinct prefix. The top log_alphabet(N) levels are saturated
# and shared by everyone; below that each term contributes its own chain.
import math
saturated_depth = math.log(distinct_terms, alphabet)
nodes = distinct_terms * (avg_term_chars - saturated_depth)
bytes_per_node_struct = 24        # compact C++ node: char + child ptr(s) + flags
K = 5
bytes_per_node_topk   = K * 8     # K x (4-byte term id + 4-byte score), cached per node

gb = lambda b: b / 1e9
print(f"Trie nodes            : {nodes/1e9:,.2f} B")
print(f"  structure only      : {gb(nodes*bytes_per_node_struct):,.0f} GB")
print(f"  + top-K at EVERY node: {gb(nodes*(bytes_per_node_struct+bytes_per_node_topk)):,.0f} GB")
print()
print("→ This does NOT fit on one box, and the top-K cache is over half of it.")
print("  Notebook 3 measures a real trie and shows the fix: cache top-K only on")
print("  shallow nodes (the hot prefixes), recompute on the fly for deep ones.")

> 🎯 Takeaway: the volume is huge. We **cannot** touch a database per keystroke.
> The hot path must be in-memory, with O(prefix length) work.
>
> Two numbers do most of the design work here:
> - **Peak QPS after the edge cache** decides how many replicas you run.
> - **Index size** decides whether you shard. At 100 M terms you do — and the
>   *top-K cache*, not the trie itself, is what pushes you over the edge.

## Input normalization — the boring-but-critical step

Before anything else, every term and every query goes through the **same** normalizer.
Skip this and `"iPhone"`, `"iphone"`, `"íphone"` all look different.

- **lowercase** (`"Google" == "google"`)
- **strip / collapse whitespace** (`"  new york "` → `"new york"`)
- **unicode fold** (`"café"` → `"cafe"`) — so `"cafe"` also finds `"café"`
- **multi-token prefixes**: users type words separated by spaces (`"new y"` should match `"new york"`)


In [ ]:
# 🧼 Minimal normalizer — we'll reuse this everywhere.
import unicodedata

def normalize(s: str) -> str:
    # NFKD splits accented chars into (letter, accent), then we drop the accents.
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = " ".join(s.split())    # collapse internal whitespace
    return s

for raw in ["Google", "  iPhone  ", "café", "NEW    York", "Pokémon GO"]:
    print(f"{raw!r:20} → {normalize(raw)!r}")


## Naïve baseline (the "bad" way) — so we feel the problem

Before we build anything clever, let's do what a beginner would:
keep all terms in a list and scan them for every keystroke.

Run it. Notice how a million-term scan already costs tens of ms —
and that's with one user on a fast laptop.


In [ ]:
# 🐢 Naive linear scan baseline.
import random, string, time

def make_corpus(n=200_000, seed=7):
    random.seed(seed)
    words = []
    for _ in range(n):
        length = random.randint(3, 12)
        words.append("".join(random.choices(string.ascii_lowercase, k=length)))
    # assign a fake popularity
    return [(w, random.randint(1, 10_000)) for w in words]

CORPUS = make_corpus()

def naive_suggest(corpus, prefix, k=5):
    prefix = normalize(prefix)
    hits = [(term, score) for term, score in corpus if term.startswith(prefix)]
    hits.sort(key=lambda x: -x[1])
    return hits[:k]

t0 = time.perf_counter()
for _ in range(20):
    naive_suggest(CORPUS, "ab", k=5)
dt = (time.perf_counter() - t0) / 20 * 1000
print(f"Naive scan over {len(CORPUS):_} terms: {dt:.2f} ms/query")
print("Sample:", naive_suggest(CORPUS, "ab", k=3))


That's for **one user on one box** with 200k terms.
Real systems have 100M+ terms and hundreds of thousands of QPS.
We need a smarter data structure — enter the **trie** (next notebook 3).


## High-level architecture

```
     [client]
        │ each keystroke
        ▼
   ┌────────┐
   │ Edge / │◀── cache popular prefixes (short prefixes repeat a lot)
   │  CDN   │
   └───┬────┘
       │ miss
       ▼
   ┌──────────┐       ┌──────────────┐
   │ Suggest  │──────▶│ Trie service │   (in-memory, replicated, sharded)
   │ Service  │       └──────┬───────┘
   └────┬─────┘              │
        │                    ▼
        │            ┌──────────────┐
        │            │ Query logs   │ (Kafka stream of every search)
        │            └──────┬───────┘
        │                   │ every N min
        │                   ▼
        │           ┌────────────────┐
        │           │ Aggregator     │  (batch job: Spark / Flink)
        │           │ → builds new   │
        │           │   trie snapshot│
        │           └───────┬────────┘
        │                   │ atomic swap
        ▼                   ▼
   return top-K       trie service loads new snapshot
```

### The two key design decisions
1. **In-memory tree** on the hot path — no DB per keystroke.
2. **Batch rebuild + atomic swap** instead of live mutations — mutating a live trie
   concurrently is hard to get right; rebuilding a fresh snapshot every 5–15 min
   is much simpler and good enough for "popularity" ranking.

Personalization and trending are overlays on top of this core (notebook 3).
